# V5W_03_read — Controllo: DHSLP su condizione READ

Stesso modello/iperparametri di V5W_03 (img), ma sui trial **`read`** (parola letta a schermo). Legge i CSV direttamente (DHSLP usa solo `x`, non i grafi → niente build di connettività).

**Scopo (controllo positivo):** se READ decodifica sopra chance e IMG no → pipeline e dati sani, il null è *specifico dell'immaginazione*. Se READ ~ chance come IMG → da indagare.

**Env: `daniele_311`**, GPU.

## §1 — Config + indice trial read

In [ ]:
import json, logging, re, traceback
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score, recall_score
from tqdm.auto import tqdm
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w03read')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'v5w03_read'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CSV_ROOT = project_root / 'data' / '5words_subjects'

# ---- CONFIG (CONTROLLO READ, 5 parole, chance 20%) ----
COND = 'read'                      # <-- controllo: lettura invece di immaginazione
N_CHANNELS, N_SAMPLES = 61, 384
N_CLASSES   = 5
_i2l = json.loads((project_root/'configs'/'label_schemes'/'idx2label_5words.json').read_text())
word2label = json.loads((project_root/'configs'/'label_schemes'/'label2idx_5words.json').read_text())
CLASS_NAMES = [_i2l[str(i)] for i in range(N_CLASSES)]

# iperparametri identici a V5W_03 (img) per confronto leale
K_WINDOWS, N_EDGES, D_MODEL, HIDDEN, N_LAYERS, DROPOUT = 8, 16, 64, 128, 2, 0.5
LR, WEIGHT_DECAY, GRAD_CLIP, BATCH_SIZE = 1e-3, 1e-2, 1.0, 32
MAX_EPOCHS, PATIENCE = 200, 70
USE_INSTANCE_NORM, LABEL_SMOOTHING, MIXUP_ALPHA = True, 0.1, 0.4
GAP_THRESHOLD, GAP_PATIENCE = 0.25, 15
USE_AUGMENTATION = True
AUG_NOISE_STD, AUG_AMP_RANGE, AUG_SHIFT_MAX, AUG_MASK_LEN = 0.05, (0.85, 1.15), 15, 20
T_WIN = N_SAMPLES // K_WINDOWS
WANDB_ENTITY, WANDB_PROJECT = 'uras-daniele22-politecnico-di-milano', 'miralis-imagined-speech'

# ---- Indice trial READ direttamente dai CSV (niente grafi: DHSLP usa solo x) ----
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
subj_sess = defaultdict(lambda: defaultdict(list))
for sd in sorted(CSV_ROOT.iterdir()):
    m = _PAT.match(sd.name)
    if not m: continue
    sid, ses = int(m.group(1)), int(m.group(2))
    for csv in sorted(sd.glob(f'*_{COND}_*.csv')):
        if csv.name.startswith('._'): continue
        word = csv.name.split(f'_{COND}_')[0]
        if word in word2label:
            subj_sess[sid][ses].append((csv, word2label[word]))
ALL_SUBJ = sorted(subj_sess)
n_trials = sum(len(v) for s in subj_sess.values() for v in s.values())
log.info(f'COND={COND}  soggetti={len(ALL_SUBJ)}  trial {COND}={n_trials}  classi={CLASS_NAMES}  chance={1/N_CLASSES:.0%}')


## §2 — Dataset (CSV read diretti)

In [ ]:
def _augment_eeg(x):
    x = x.clone()
    if torch.rand(1) < 0.5: x = x + torch.randn_like(x) * AUG_NOISE_STD
    if torch.rand(1) < 0.5: x = x * torch.empty(1).uniform_(*AUG_AMP_RANGE)
    if torch.rand(1) < 0.5:
        sh = torch.randint(-AUG_SHIFT_MAX, AUG_SHIFT_MAX + 1, (1,)).item(); x = torch.roll(x, sh, dims=1)
    if torch.rand(1) < 0.3:
        T = x.shape[1]; st = torch.randint(0, max(1, T - AUG_MASK_LEN), (1,)).item(); x[:, st:st+AUG_MASK_LEN] = 0.0
    if torch.rand(1) < 0.2:
        ch = torch.randint(0, x.shape[0], (1,)).item(); x[ch] = 0.0
    return x

class CSVDatasetSS(Dataset):
    """Legge i CSV (61,384) direttamente, instance-norm, label = parola 0-4."""
    def __init__(self, items, augment=False):
        self.items, self.augment = items, augment
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        x = torch.from_numpy(pd.read_csv(p, header=None).values.astype(np.float32))
        if USE_INSTANCE_NORM:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        if self.augment: x = _augment_eeg(x)
        return x, torch.tensor(label, dtype=torch.long)

def _collect(sid, sess_list):
    items = []
    for s in sess_list: items += subj_sess[sid][s]
    return items

def make_loso_loaders(sid):
    sids = sorted(subj_sess[sid].keys())
    if len(sids) < 2: return None
    te_s, va_s = sids[-1], sids[-2]
    tr_s = [s for s in sids if s not in (te_s, va_s)]
    tr_i, va_i, te_i = _collect(sid, tr_s), _collect(sid, [va_s]), _collect(sid, [te_s])
    if not tr_i or not te_i: return None
    lbl = np.array([it[1] for it in tr_i]); cnt = np.bincount(lbl, minlength=N_CLASSES)
    w = torch.tensor(1.0/np.clip(cnt[lbl],1,None), dtype=torch.float)
    samp = WeightedRandomSampler(w, len(w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(CSVDatasetSS(tr_i, USE_AUGMENTATION), BATCH_SIZE, sampler=samp, **kw),
            DataLoader(CSVDatasetSS(va_i, False), BATCH_SIZE, shuffle=False, **kw),
            DataLoader(CSVDatasetSS(te_i, False), BATCH_SIZE, shuffle=False, **kw))


## §3 — Modello DHSLP

In [ ]:
class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)
    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6); d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv = (1.0/d_v.sqrt()).unsqueeze(-1); De = (1.0/d_e).unsqueeze(1)
        out = Dv * (X @ self.weight); out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out; out = torch.bmm(H, out); out = Dv * out
        if self.bias is not None: out = out + self.bias
        return out

class DHSLP(nn.Module):
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS, n_edges=N_EDGES,
                 d_model=D_MODEL, hidden=HIDDEN, n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(nn.Linear(T_win, d_model), nn.LayerNorm(d_model), nn.ELU())
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout); self.clf = nn.Linear(hidden, n_classes)
    def build_dynamic_H(self, feat):
        return torch.softmax(torch.matmul(feat, self.E.T) / (self.d_model ** 0.5), dim=2)
    def forward(self, x):
        B, N, T = x.shape; outs = []
        for k in range(self.K):
            x_k = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc
            H_k = self.build_dynamic_H(feat); out = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k); out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out); out = self.drop(out)
            outs.append(out.mean(dim=1))
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_m = DHSLP(); log.info(f'device={device}  DHSLP {sum(p.numel() for p in _m.parameters()):,} params'); del _m


## §4 — Train / Eval

In [ ]:
_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def run_epoch(model, loader, optimizer=None, mixup_alpha=0.0):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot, L, P = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train and mixup_alpha > 0:
                lam = float(np.random.beta(mixup_alpha, mixup_alpha)); idx = torch.randperm(len(x), device=x.device)
                x = lam*x + (1-lam)*x[idx]; logits = model(x)
                loss = lam*_criterion(logits, y) + (1-lam)*_criterion(logits, y[idx])
            else:
                logits = model(x); loss = _criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); optimizer.step()
            tot += loss.item()*len(y); L.extend(y.cpu().numpy()); P.extend(logits.argmax(1).cpu().numpy())
    return tot/len(loader.dataset), balanced_accuracy_score(L, P), np.array(L), np.array(P)

def train_subject(sid, tr_l, va_l, te_l):
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f'v5w03read_P{sid:03d}_5words',
                     config=dict(notebook='V5W_03_read', model='DHSLP_SS_read', cond=COND, subject=f'P{sid:03d}',
                                 n_classes=N_CLASSES, max_epochs=MAX_EPOCHS, n_train=len(tr_l.dataset)),
                     reinit='finish_previous', settings=wandb.Settings(start_method='thread'))
    model = DHSLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, pc, gc = 0.0, None, 0, 0
    for ep in range(1, MAX_EPOCHS + 1):
        _, tr_b, _, _ = run_epoch(model, tr_l, opt, mixup_alpha=MIXUP_ALPHA)
        _, va_b, _, _ = run_epoch(model, va_l); sched.step(); gap = tr_b - va_b
        run.log({'train/bacc': tr_b, 'val/bacc': va_b, 'train_val_gap': gap, 'epoch': ep})
        if va_b > best_val: best_val = va_b; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; pc = 0
        else: pc += 1
        if pc >= PATIENCE: break
        if gap > GAP_THRESHOLD:
            gc += 1
            if gc >= GAP_PATIENCE: break
        else: gc = 0
    model.load_state_dict(best_state)
    _, te_b, te_lbl, te_pred = run_epoch(model, te_l)
    torch.save({'state_dict': best_state, 'val_bacc': best_val, 'test_bacc': te_b,
                'labels': te_lbl, 'preds': te_pred}, CKPT_DIR / f'P{sid:03d}.pt')
    run.summary['val_bacc'], run.summary['test_bacc'] = best_val, te_b
    try:
        run.log({'confusion_matrix': wandb.plot.confusion_matrix(
            preds=te_pred.tolist(), y_true=te_lbl.tolist(), class_names=CLASS_NAMES)})
    except Exception: pass
    run.finish()
    return {'val_bacc': best_val, 'test_bacc': te_b}


## §5 — Loop soggetti

In [ ]:
SUBJECT_RESULTS = {}
for sid in tqdm(ALL_SUBJ, desc=f'DHSLP SS {COND}'):
    if (CKPT_DIR / f'P{sid:03d}.pt').exists():
        log.info(f'P{sid:03d}: skip (checkpoint)'); continue
    loaders = make_loso_loaders(sid)
    if loaders is None:
        log.warning(f'P{sid:03d}: skip (sessioni insufficienti)'); continue
    tr_l, va_l, te_l = loaders
    log.info(f'P{sid:03d}: train={len(tr_l.dataset)} val={len(va_l.dataset)} test={len(te_l.dataset)}')
    try:
        SUBJECT_RESULTS[sid] = train_subject(sid, tr_l, va_l, te_l)
    except Exception as e:
        log.error(f'P{sid:03d}: {e}\n{traceback.format_exc()}')
log.info(f'DONE: {len(SUBJECT_RESULTS)}/{len(ALL_SUBJ)}')


## §6 — Ranking READ + confronto IMG

In [ ]:
# Ranking READ + confronto col controllo IMG (V5W_03)
rows = []
for ck in sorted(CKPT_DIR.glob('P*.pt')):
    d = torch.load(ck, weights_only=False); rows.append((ck.stem, float(d['test_bacc'])))
df = pd.DataFrame(rows, columns=['Subject', 'Test bAcc']).sort_values('Test bAcc', ascending=False).reset_index(drop=True)
chance = 1/N_CLASSES; b = df['Test bAcc'].values
print('='*50); print(f'  V5W_03_read — DHSLP SS condizione READ ({len(b)} sogg.)'); print('='*50)
print(f'  mean={b.mean():.4f}  median={np.median(b):.4f}  max={b.max():.4f}  chance={chance:.3f}')
print(f'  > chance: {(b>chance).sum()}/{len(b)} ({(b>chance).mean()*100:.1f}%)')

# confronto con IMG se i checkpoint esistono
img_dir = project_root / 'models' / 'v5w03'
b_img = np.array([float(torch.load(c, weights_only=False)['test_bacc']) for c in sorted(img_dir.glob('P*.pt'))]) if img_dir.exists() else np.array([])
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(b)), b, color=['#2ca02c' if v>chance else '#d62728' for v in b], alpha=0.85)
ax.axhline(chance, color='k', ls='--', lw=1.5, label=f'Chance ({chance:.0%})')
if len(b_img):
    ax.axhline(b_img.mean(), color='#1f77b4', ls=':', lw=2, label=f'IMG mean ({b_img.mean():.3f})')
ax.axhline(b.mean(), color='#2ca02c', ls=':', lw=2, label=f'READ mean ({b.mean():.3f})')
ax.set_xticks(range(len(b))); ax.set_xticklabels(df['Subject'], rotation=90, fontsize=6)
ax.set_ylabel('Balanced Accuracy'); ax.set_title('V5W_03 READ vs chance (e media IMG)'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w03_read_ranking.png', dpi=150, bbox_inches='tight'); plt.show()
if len(b_img):
    from scipy.stats import mannwhitneyu
    u, p = mannwhitneyu(b, b_img, alternative='greater')
    print(f'\n  READ mean={b.mean():.4f}  vs  IMG mean={b_img.mean():.4f}')
    print(f'  Mann-Whitney READ>IMG: p={p:.4f}')
    print('  → READ >> IMG e sopra chance: pipeline OK, il null e\\\' specifico dell\\\'immaginazione.')
    print('  → READ ~ IMG ~ chance: nessun segnale di parola nemmeno in lettura (da indagare).')
